# 05 — Build-time Derivations

Verify that `build_model` fills in derivable tables (`periods`,
`facility_roles`, `demand`, `supply`, `inventory_initial`, default
category tables) when the loader produces a minimal `RawModelData`,
and that each derivation is recorded in `resolved.build_report`.

Also checks the manual-wins rule: a user-provided table is never
overwritten by a derivation.

In [1]:
from gbp.build.pipeline import build_model
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

mock = DataLoaderMock({'n_stations': 10, 'n_depots': 2, 'n_timestamps': 168})
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved = build_model(raw)

# The bike-sharing loader now emits a minimal raw contract: no periods,
# no facility_roles, no demand/supply — all derived during build.
# (inventory_initial is still computed from telemetry at load time.)
assert raw.periods is None
assert raw.facility_roles is None
assert raw.demand is None
assert raw.supply is None

assert resolved.build_report is not None
resolved.build_report.derivations


2026-04-15 21:27:04 [info     ] load_start                     loader=graph_core
2026-04-15 21:27:05 [debug    ] source_validated               loader=graph_core
2026-04-15 21:27:05 [debug    ] observed_flow_built            loader=graph_core rows=727
2026-04-15 21:27:05 [debug    ] observed_inventory_built       loader=graph_core rows=140
2026-04-15 21:27:05 [info     ] load_done                      facilities=12 loader=graph_core


{'periods': 'built from planning_horizon_segments',
 'facility_roles': 'derived from facility_type + facility_operations via derive_roles()',
 'demand': 'derived from observed_flow (groupby source_id × date × cc)',
 'supply': 'derived from observed_flow (groupby target_id × date × cc)'}

In [2]:
print(f"resolved.periods:           {len(resolved.periods)} rows")
print(f"resolved.facility_roles:    {len(resolved.facility_roles)} rows")
print(f"resolved.demand:            {len(resolved.demand)} rows")
print(f"resolved.supply:            {len(resolved.supply)} rows")
print(f"resolved.inventory_initial: {len(resolved.inventory_initial)} rows")

resolved.periods:           7 rows
resolved.facility_roles:    44 rows
resolved.demand:            138 rows
resolved.supply:            140 rows
resolved.inventory_initial: 24 rows


In [3]:
# Manual-wins: when the user provides a table, derivation is skipped.
import dataclasses
from gbp.build.pipeline import build_model
from tests.unit.build.fixtures import minimal_raw_model

full_raw = minimal_raw_model()
full_resolved = build_model(full_raw)
assert full_resolved.build_report.derivations == {}
full_resolved.build_report.derivations

{}

In [4]:
# Drop just `periods` from a full raw → only `periods` is re-derived.
partial_raw = dataclasses.replace(full_raw, periods=None)
partial_resolved = build_model(partial_raw)
partial_resolved.build_report.derivations

{'periods': 'built from planning_horizon_segments'}

In [5]:
# `Environment.__init__` raises when no flow inputs exist.
from gbp.consumers.simulator.config import EnvironmentConfig
from gbp.consumers.simulator.engine import Environment
from gbp.consumers.simulator.exceptions import SimulatorConfigError

empty_resolved = build_model(minimal_raw_model(with_demand=False, with_supply=False))
empty_resolved.demand = None
empty_resolved.supply = None
empty_resolved.inventory_initial = None

try:
    Environment(empty_resolved, EnvironmentConfig(phases=[]))
    raised = False
except SimulatorConfigError as exc:
    raised = True
    message = str(exc)

assert raised
message

'Environment requires demand, supply, or inventory_initial. Provide them in RawModelData, or provide observed_flow / observed_inventory so build_model() can derive them.'